# Módulo 8 · Clase 9 — Entrenamiento y Monitorización

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | TensorBoard: visualización del entrenamiento |
| 2 | Learning Rate Schedules |
| 3 | Callbacks personalizados |
| 4 | Diagnóstico avanzado de modelos |
| 5 | Flujo de trabajo profesional |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os, datetime

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

# Dataset base para los ejemplos
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.mnist.load_data()
X_tr = X_tr_raw.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te_raw.reshape(-1, 784).astype('float32') / 255.0
print(f"MNIST cargado: {X_tr.shape}")

---
## 1. TensorBoard: visualización del entrenamiento

```
TensorBoard es la herramienta de visualización de TensorFlow.
Permite ver en tiempo real (o tras el entrenamiento):
  - Curvas de loss y métricas
  - Histogramas de pesos y gradientes por capa
  - Grafos del modelo
  - Imágenes y audio de muestra
  - Proyección de embeddings en 3D
  - Profiling de rendimiento

Uso:
  1. Añadir callback TensorBoard al entrenamiento
  2. Lanzar: tensorboard --logdir logs/
  3. Abrir http://localhost:6006 en el navegador

En Jupyter:
  %load_ext tensorboard
  %tensorboard --logdir logs/
```

In [ ]:
# Configurar TensorBoard
log_dir = 'logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
print(f"Logs en: {log_dir}")

tf.random.set_seed(42)
modelo_tb = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu', name='capa1'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu', name='capa2'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax', name='salida'),
])
modelo_tb.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

tb_callback = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,      # histogramas de pesos cada N épocas
    write_graph=True,      # guardar el grafo del modelo
    write_images=False,
    update_freq='epoch',   # frecuencia de log: 'epoch' o 'batch'
    profile_batch=0,       # 0 = desactivar profiling
)

hist_tb = modelo_tb.fit(
    X_tr, y_tr_raw,
    epochs=10, batch_size=256,
    validation_split=0.1,
    callbacks=[tb_callback],
    verbose=0
)

_, acc = modelo_tb.evaluate(X_te, y_te_raw, verbose=0)
print(f"Test accuracy: {acc:.4f}")
print(f"\nPara ver TensorBoard, ejecuta en terminal:")
print(f"  tensorboard --logdir {os.path.dirname(log_dir)}")
print("Luego abre http://localhost:6006")

In [ ]:
# Escribir métricas personalizadas en TensorBoard
log_custom = 'logs/custom/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
writer = tf.summary.create_file_writer(log_custom)

# Simular entrenamiento manual y log de métricas
tf.random.set_seed(42)
m_manual = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
optimizer = keras.optimizers.Adam(0.001)
loss_fn   = keras.losses.SparseCategoricalCrossentropy()

dataset = tf.data.Dataset.from_tensor_slices((X_tr, y_tr_raw)).batch(256)

for epoch in range(5):
    losses = []
    for xb, yb in dataset:
        with tf.GradientTape() as tape:
            logits = m_manual(xb, training=True)
            loss   = loss_fn(yb, logits)
        grads = tape.gradient(loss, m_manual.trainable_variables)
        optimizer.apply_gradients(zip(grads, m_manual.trainable_variables))
        losses.append(float(loss))

    mean_loss = np.mean(losses)
    # Escribir en TensorBoard
    with writer.as_default():
        tf.summary.scalar('train_loss_manual', mean_loss, step=epoch)
        tf.summary.scalar('lr', float(optimizer.learning_rate), step=epoch)
    print(f"Época {epoch+1}: loss={mean_loss:.4f}")

print(f"\nLogs custom en: {log_custom}")

---
## 2. Learning Rate Schedules

In [ ]:
print("""
La tasa de aprendizaje (lr) es el hiperparámetro más importante.
Un lr fijo raramente es lo óptimo:

  Alta al inicio → explora el espacio, desciende rápido
  Baja al final  → afina la convergencia

Estrategias de lr schedule:

  1. Step Decay:           lr × factor cada N épocas
  2. Exponential Decay:    lr × e^(-k·t)
  3. Cosine Annealing:     lr oscila como coseno
  4. Warmup + Decay:       sube gradualmente, luego baja (Transformers)
  5. OneCycleLR:           lr_max en el medio (muy efectivo en la práctica)
  6. ReduceLROnPlateau:    ya visto — reduce si val_loss estanca
""")

In [ ]:
# Implementar y comparar schedules
n_epocas = 50
epocas   = np.arange(n_epocas)

def step_decay(epoca, lr_inicial=0.01, factor=0.5, paso=10):
    return lr_inicial * (factor ** (epoca // paso))

def exp_decay(epoca, lr_inicial=0.01, k=0.05):
    return lr_inicial * np.exp(-k * epoca)

def cosine_annealing(epoca, lr_max=0.01, lr_min=1e-5, T=50):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoca / T))

def warmup_decay(epoca, lr_max=0.01, warmup=10, T=50):
    if epoca < warmup:
        return lr_max * (epoca + 1) / warmup
    return lr_max * np.exp(-0.05 * (epoca - warmup))

schedules = {
    'Step Decay':        [step_decay(e) for e in epocas],
    'Exp Decay':         [exp_decay(e) for e in epocas],
    'Cosine Annealing':  [cosine_annealing(e) for e in epocas],
    'Warmup + Decay':    [warmup_decay(e) for e in epocas],
    'Constante (0.001)': [0.001] * n_epocas,
}

fig, ax = plt.subplots(figsize=(12, 5))
for nombre, lr_vals in schedules.items():
    ax.semilogy(epocas, lr_vals, lw=2, label=nombre)
ax.set_xlabel('Época'); ax.set_ylabel('Learning rate (escala log)')
ax.set_title('Comparación de Learning Rate Schedules')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

In [ ]:
# Aplicar schedules en Keras
# Opción 1: LearningRateScheduler callback
def schedule_fn(epoca):
    if epoca < 10:
        return 0.001 * (epoca + 1) / 10   # warmup
    return 0.001 * np.exp(-0.03 * (epoca - 10))

lr_callback = keras.callbacks.LearningRateScheduler(schedule_fn, verbose=0)

# Opción 2: keras.optimizers.schedules
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001,
    decay_steps=10000,
    alpha=1e-5  # lr mínima
)

# Comparar: lr fija vs warmup+decay
resultados_lr = {}
for nombre, optimizer in [
    ('Adam lr=0.001 fija', keras.optimizers.Adam(0.001)),
    ('CosineDecay',        keras.optimizers.Adam(lr_schedule)),
]:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64,  activation='relu'),
        layers.Dense(10,  activation='softmax'),
    ])
    m.compile(optimizer, 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr, y_tr_raw, epochs=15, batch_size=256,
              validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te, y_te_raw, verbose=0)
    resultados_lr[nombre] = (h.history['val_accuracy'], acc)
    print(f"{nombre}: val_acc final={h.history['val_accuracy'][-1]:.4f}, test_acc={acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
for nombre, (val_acc, _) in resultados_lr.items():
    ax.plot(val_acc, lw=2, label=nombre)
ax.set_xlabel('Época'); ax.set_ylabel('Val Accuracy')
ax.set_title('Efecto del LR Schedule')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3. Callbacks personalizados

In [ ]:
# Los callbacks de Keras se implementan sobrescribiendo métodos

class MonitorAvanzado(keras.callbacks.Callback):
    """Callback que monitoriza, detecta problemas y guarda el mejor modelo."""

    def __init__(self, umbral_gap=0.05, ruta_guardado='mejor.keras'):
        super().__init__()
        self.umbral_gap    = umbral_gap
        self.ruta          = ruta_guardado
        self.mejor_val_acc = 0.0
        self.historia      = {'loss':[], 'val_loss':[], 'acc':[], 'val_acc':[], 'gap':[], 'lr':[]}

    def on_train_begin(self, logs=None):
        print(f"[Monitor] Entrenamiento iniciado. Gap máximo permitido: {self.umbral_gap}")

    def on_epoch_end(self, epoch, logs=None):
        acc     = logs.get('accuracy',     0)
        val_acc = logs.get('val_accuracy', 0)
        loss    = logs.get('loss',         0)
        val_loss= logs.get('val_loss',     0)
        gap     = acc - val_acc
        lr      = float(self.model.optimizer.learning_rate)

        for k, v in [('loss',loss),('val_loss',val_loss),('acc',acc),
                     ('val_acc',val_acc),('gap',gap),('lr',lr)]:
            self.historia[k].append(v)

        # Guardar si mejora
        if val_acc > self.mejor_val_acc:
            self.mejor_val_acc = val_acc
            self.model.save(self.ruta)
        # Alerta de overfitting
        if gap > self.umbral_gap and epoch > 5:
            print(f"  [Monitor] ALERTA overfitting: gap={gap:.4f} (época {epoch+1})")
        # Log compacto cada 5 épocas
        if (epoch + 1) % 5 == 0:
            print(f"  [Monitor] Época {epoch+1:3d}: loss={loss:.4f} val={val_loss:.4f} "
                  f"acc={acc:.4f} val_acc={val_acc:.4f} gap={gap:.4f}")

    def on_train_end(self, logs=None):
        print(f"[Monitor] Mejor val_accuracy: {self.mejor_val_acc:.4f}")
        self.plot()

    def plot(self):
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].plot(self.historia['loss'],     'steelblue', label='Train')
        axes[0].plot(self.historia['val_loss'], 'coral',     label='Val')
        axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(self.historia['acc'],     'steelblue', label='Train')
        axes[1].plot(self.historia['val_acc'], 'coral',     label='Val')
        axes[1].fill_between(range(len(self.historia['gap'])),
                              self.historia['val_acc'],
                              self.historia['acc'], alpha=0.2, color='red', label='Gap')
        axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].semilogy(self.historia['lr'], 'purple', lw=2)
        axes[2].set_title('Learning rate'); axes[2].grid(True, alpha=0.3, which='both')

        for ax in axes:
            ax.set_xlabel('Época')
        plt.tight_layout(); plt.show()


# Usar el callback
tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

monitor = MonitorAvanzado(umbral_gap=0.03)
m.fit(X_tr, y_tr_raw, epochs=20, batch_size=256,
      validation_split=0.1, callbacks=[monitor], verbose=0)

---
## 4. Diagnóstico avanzado de modelos

In [ ]:
# Monitorizar el flujo de gradientes por capa
# Problema: gradientes muy pequeños → capa no aprende
#           gradientes muy grandes → inestabilidad

tf.random.set_seed(42)
m_diag = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu', name='d1'),
    layers.Dense(256, activation='relu', name='d2'),
    layers.Dense(128, activation='relu', name='d3'),
    layers.Dense(64,  activation='relu', name='d4'),
    layers.Dense(10,  activation='softmax', name='salida'),
])

optimizer = keras.optimizers.Adam(0.001)
loss_fn   = keras.losses.SparseCategoricalCrossentropy()

X_batch = tf.constant(X_tr[:256])
y_batch = tf.constant(y_tr_raw[:256])

with tf.GradientTape() as tape:
    pred = m_diag(X_batch, training=True)
    loss = loss_fn(y_batch, pred)

grads = tape.gradient(loss, m_diag.trainable_variables)

print("Flujo de gradientes por capa:")
print(f"{'Capa':15} {'Shape':20} {'media |grad|':>14} {'max |grad|':>12}")
print('-' * 65)
for var, grad in zip(m_diag.trainable_variables, grads):
    if grad is not None and 'kernel' in var.name:
        g = np.abs(grad.numpy())
        print(f"{var.name[:14]:15} {str(var.shape):20} {g.mean():>14.6f} {g.max():>12.6f}")

# Visualizar
grad_means = []
for var, grad in zip(m_diag.trainable_variables, grads):
    if grad is not None and 'kernel' in var.name:
        grad_means.append((var.name.split('/')[0], np.abs(grad.numpy()).mean()))

nombres, medias = zip(*grad_means)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(nombres, medias, color='steelblue')
ax.set_ylabel('Media |gradiente|'); ax.set_title('Flujo de gradientes (antes del entrenamiento)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

In [ ]:
# Distribución de pesos antes y después de entrenar
m_before = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

pesos_antes = m_before.layers[0].get_weights()[0].ravel()

m_before.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_before.fit(X_tr, y_tr_raw, epochs=5, batch_size=256, verbose=0)

pesos_despues = m_before.layers[0].get_weights()[0].ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pesos_antes,   bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Pesos antes (mean={pesos_antes.mean():.4f}, std={pesos_antes.std():.4f})')
axes[0].set_xlabel('Valor del peso')

axes[1].hist(pesos_despues, bins=60, color='coral',     edgecolor='white', alpha=0.8)
axes[1].set_title(f'Pesos después (mean={pesos_despues.mean():.4f}, std={pesos_despues.std():.4f})')
axes[1].set_xlabel('Valor del peso')

plt.suptitle('Distribución de pesos: capa Dense(128) antes y después de entrenar', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Detectar 'neuronas muertas' (Dead ReLU)
# Una neurona ReLU 'muere' si siempre produce 0 → no contribuye al aprendizaje

tf.random.set_seed(42)
m_relu = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu', name='capa_relu'),
    layers.Dense(10, activation='softmax'),
])
m_relu.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_relu.fit(X_tr, y_tr_raw, epochs=10, batch_size=256, verbose=0)

# Calcular activaciones en un batch grande
m_relu_interm = keras.Model(inputs=m_relu.input,
                             outputs=m_relu.get_layer('capa_relu').output)
activaciones  = m_relu_interm.predict(X_te, batch_size=512, verbose=0)

n_neuronas_total  = activaciones.shape[1]
n_muestras_test   = activaciones.shape[0]

# Neurona 'muerta' si activa en <1% de las muestras de test
tasa_activacion = (activaciones > 0).mean(axis=0)
n_muertas       = (tasa_activacion < 0.01).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(tasa_activacion, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0.01, color='red', ls='--', label='Umbral muerta (1%)')
axes[0].set_xlabel('Fracción de muestras donde activa'); axes[0].set_ylabel('Neuronas')
axes[0].set_title(f'Distribución de activación — {n_muertas}/{n_neuronas_total} muertas')
axes[0].legend()

# Activaciones medias por neurona
axes[1].bar(range(n_neuronas_total), np.sort(tasa_activacion),
            color=['red' if t < 0.01 else 'steelblue' for t in np.sort(tasa_activacion)])
axes[1].axhline(0.01, color='red', ls='--', lw=1)
axes[1].set_xlabel('Neurona (ordenada)'); axes[1].set_ylabel('Tasa de activación')
axes[1].set_title('Tasa de activación por neurona (rojo = muerta)')

plt.tight_layout(); plt.show()
print(f"Neuronas muertas: {n_muertas}/{n_neuronas_total} ({n_muertas/n_neuronas_total:.1%})")

---
## 5. Flujo de trabajo profesional

In [ ]:
# Flujo de trabajo completo para un proyecto de DL
from sklearn.metrics import classification_report

print("""
╔═══════════════════════════════════════════════════════════╗
║         FLUJO DE TRABAJO PROFESIONAL EN DEEP LEARNING     ║
╠═══════════════════════════════════════════════════════════╣
║ 1. Preparar datos         → normalizar, split, augmentar  ║
║ 2. Baseline simple        → Dense pequeño, pocas épocas   ║
║ 3. Aumentar complejidad   → más capas, más neuronas        ║
║ 4. Regularizar            → Dropout, BN, L2               ║
║ 5. Afinar lr y schedule   → CosineDecay, Warmup           ║
║ 6. Monitorizar            → TensorBoard, callbacks         ║
║ 7. Diagnosticar           → curvas, gradientes, pesos     ║
║ 8. Evaluar en test        → solo al final, una sola vez   ║
║ 9. Guardar y documentar   → .keras + JSON de config       ║
╚═══════════════════════════════════════════════════════════╝
""")

# Ejemplo completo con todas las buenas prácticas
def entrenar_modelo_profesional(X_tr, y_tr, X_te, y_te,
                                 nombre='modelo', log_dir='logs'):
    """Pipeline profesional de entrenamiento."""
    import json

    # 1. Arquitectura
    tf.random.set_seed(42)
    modelo = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax'),
    ], name=nombre)

    # 2. LR schedule
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001, decay_steps=len(X_tr)//256*20
    )

    # 3. Compilar
    modelo.compile(
        optimizer=keras.optimizers.Adam(lr_schedule),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # 4. Callbacks
    ts = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    cbs = [
        keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(f'{nombre}_best.keras', save_best_only=True, verbose=0),
        keras.callbacks.TensorBoard(f'{log_dir}/{nombre}/{ts}', histogram_freq=1),
    ]

    # 5. Entrenar
    hist = modelo.fit(
        X_tr, y_tr,
        epochs=30, batch_size=256,
        validation_split=0.1,
        callbacks=cbs,
        verbose=0
    )

    # 6. Evaluar en test
    loss_te, acc_te = modelo.evaluate(X_te, y_te, verbose=0)
    n_epocas = len(hist.history['loss'])

    # 7. Guardar metadatos
    config = {
        'nombre':         nombre,
        'timestamp':      ts,
        'n_epocas':       n_epocas,
        'val_acc_final':  hist.history['val_accuracy'][-1],
        'test_loss':      float(loss_te),
        'test_accuracy':  float(acc_te),
        'parametros':     modelo.count_params(),
    }
    with open(f'{nombre}_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    print(f"\n[{nombre}]")
    print(f"  Épocas: {n_epocas}  Parámetros: {modelo.count_params():,}")
    print(f"  Val accuracy: {hist.history['val_accuracy'][-1]:.4f}")
    print(f"  Test accuracy: {acc_te:.4f}")
    return modelo, hist, config

modelo_final, hist_final, config = entrenar_modelo_profesional(
    X_tr, y_tr_raw, X_te, y_te_raw, nombre='mnist_profesional'
)

print("\nConfig guardada en mnist_profesional_config.json")

---
## Práctica

### Ejercicio 1 — TensorBoard comparativo

**Enunciado:** TensorBoard permite comparar múltiples experimentos en la misma pantalla.

1. Entrena 3 modelos con distintas arquitecturas en Fashion MNIST
2. Guarda cada uno en un subdirectorio distinto de `logs/experimentos/`
3. Lanza TensorBoard apuntando a `logs/experimentos/`
4. Compara las curvas de loss en la misma gráfica

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import datetime

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configs = [
    ('pequeño_32',   [32]),
    ('mediano_128',  [128, 64]),
    ('grande_256',   [256, 128, 64]),
]

for nombre, capas_h in configs:
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in capas_h:
        capas.extend([layers.Dense(n, activation='relu'), layers.Dropout(0.3)])
    capas.append(layers.Dense(10, activation='softmax'))
    m = keras.Sequential(capas, name=nombre)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

    ts = datetime.datetime.now().strftime('%H%M%S')
    tb = keras.callbacks.TensorBoard(f'logs/experimentos/{nombre}_{ts}', histogram_freq=0)
    m.fit(X_tr, y_tr, epochs=10, batch_size=256, validation_split=0.1,
          callbacks=[tb], verbose=0)
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    print(f"{nombre}: {m.count_params():,} params, acc={acc:.4f}")

print("\nLanza: tensorboard --logdir logs/experimentos")
```
</details>

---
### Ejercicio 2 — WarmupCosineDecay desde cero

**Enunciado:** Implementa un schedule de lr personalizado como subclase de `keras.optimizers.schedules.LearningRateSchedule`:

1. Primeras `warmup_steps` pasos: lr sube linealmente de 0 a `peak_lr`
2. Después: cosine decay hasta `min_lr`
3. Úsalo en un modelo MNIST y visualiza la curva de lr durante el entrenamiento

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.peak_lr      = peak_lr
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.min_lr       = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        decay_progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = self.min_lr + 0.5*(self.peak_lr - self.min_lr)*(1 + tf.cos(np.pi * decay_progress))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {'peak_lr': self.peak_lr, 'warmup_steps': int(self.warmup_steps.numpy()),
                'total_steps': int(self.total_steps.numpy()), 'min_lr': self.min_lr}

# Visualizar el schedule
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

N_EPOCAS = 15; BATCH = 256
pasos_por_epoca = len(X_tr) // BATCH
total_steps  = N_EPOCAS * pasos_por_epoca
warmup_steps = 2 * pasos_por_epoca  # 2 épocas de warmup

schedule = WarmupCosineDecay(peak_lr=0.002, warmup_steps=warmup_steps, total_steps=total_steps)

pasos = np.arange(total_steps)
lrs   = [float(schedule(p)) for p in pasos]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pasos, lrs, 'steelblue', lw=2)
ax.axvline(warmup_steps, color='red', ls='--', label=f'Warmup ({warmup_steps} pasos)')
ax.set_xlabel('Paso'); ax.set_ylabel('Learning rate')
ax.set_title('WarmupCosineDecay')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Usar en el modelo
tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m.compile(keras.optimizers.Adam(schedule), 'sparse_categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr, epochs=N_EPOCAS, batch_size=BATCH, validation_split=0.1, verbose=0)
_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Test accuracy: {acc:.4f}")
```
</details>

---
### Ejercicio 3 — Callback de prueba de estrés

**Enunciado:** Crea un callback que, cada 5 épocas, evalúe el modelo en un conjunto de 'muestras difíciles' (las que tiene más baja confianza) y las registre:

1. `on_epoch_end`: si `epoch % 5 == 0`, evalúa en X_te, detecta las 20 muestras con menor confianza
2. Guarda un log de cuántas de esas 20 están correctas
3. Al final del entrenamiento, plotea cómo evoluciona ese número

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

class PruebaEstres(keras.callbacks.Callback):
    def __init__(self, X_te, y_te, n_dificiles=20, frecuencia=5):
        super().__init__()
        self.X_te        = X_te
        self.y_te        = y_te
        self.n           = n_dificiles
        self.frecuencia  = frecuencia
        self.correctas   = []  # por control
        self.epocas_log  = []

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.frecuencia != 0:
            return
        probas    = self.model.predict(self.X_te, verbose=0)
        max_proba = probas.max(axis=1)  # confianza máxima por muestra
        idx_dif   = np.argsort(max_proba)[:self.n]  # las n con menor confianza
        y_pred    = probas[idx_dif].argmax(axis=1)
        n_ok      = (y_pred == self.y_te[idx_dif]).sum()
        self.correctas.append(n_ok)
        self.epocas_log.append(epoch + 1)
        print(f"  [Estrés época {epoch+1}] {n_ok}/{self.n} correctas entre las más difíciles")

    def on_train_end(self, logs=None):
        if not self.epocas_log:
            return
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(self.epocas_log, self.correctas, 'o-', color='steelblue', lw=2)
        ax.set_xlabel('Época'); ax.set_ylabel(f'Correctas de las {self.n} más difíciles')
        ax.set_title('Evolución del rendimiento en muestras difíciles')
        ax.set_ylim(0, self.n + 1); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m.fit(X_tr, y_tr, epochs=25, batch_size=256, validation_split=0.1,
      callbacks=[PruebaEstres(X_te, y_te)], verbose=0)
```
</details>

---
### Ejercicio 4 — Análisis de errores por clase

**Enunciado:** No todos los errores son iguales. Analiza los errores del modelo MNIST por clase:

1. Calcula precision, recall y F1 por dígito
2. Identifica los 5 pares de dígitos que se confunden más
3. Visualiza las 3 muestras más erróneas de cada par confuso
4. ¿Qué patrones visuales explican las confusiones?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

y_pred = modelo_final.predict(X_te, verbose=0).argmax(axis=1)

print("Reporte por dígito:")
print(classification_report(y_te_raw, y_pred, target_names=[str(i) for i in range(10)]))

# Pares más confusos (excluyendo diagonal)
cm = confusion_matrix(y_te_raw, y_pred)
np.fill_diagonal(cm, 0)

# Top 5 pares
top5 = []
cm_copy = cm.copy()
for _ in range(5):
    idx = np.unravel_index(cm_copy.argmax(), cm_copy.shape)
    top5.append((idx[0], idx[1], cm_copy[idx]))
    cm_copy[idx] = 0

print("\nTop 5 confusiones:")
for real, pred, n in top5:
    print(f"  {real} → {pred}: {n} veces")

# Visualizar muestras confusas
fig, axes = plt.subplots(len(top5), 3, figsize=(8, len(top5)*2.5))
for row, (real, pred, n) in enumerate(top5):
    # Muestras donde real=real y predicho=pred
    mask = (y_te_raw == real) & (y_pred == pred)
    idx_err = np.where(mask)[0][:3]
    for col, i in enumerate(idx_err):
        ax = axes[row, col]
        ax.imshow(X_te_raw[i], cmap='gray')
        if col == 0:
            ax.set_ylabel(f'{real}→{pred} ({n})', fontsize=9, rotation=0, labelpad=40, va='center')
        ax.set_title(f'Real:{real} Pred:{pred}', fontsize=8, color='red')
        ax.axis('off')
plt.suptitle('Muestras de los 5 pares más confusos', fontsize=13)
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 5 (reto) — Experimento automatizado

**Enunciado:** Construye un sistema de experimentos que entrene varias configuraciones, guarde los resultados y seleccione automáticamente la mejor:

1. Define una lista de 4 configuraciones (batch_size, dropout, lr)
2. Entrena cada una en Fashion MNIST con 10 épocas
3. Guarda los resultados en un CSV (`experimentos.csv`)
4. Carga el CSV con pandas, ordena por val_acc y reentrena la mejor configuración 30 épocas
5. Plóta la tabla de resultados

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0

configuraciones = [
    {'batch': 64,  'dropout': 0.2, 'lr': 0.001},
    {'batch': 128, 'dropout': 0.3, 'lr': 0.001},
    {'batch': 256, 'dropout': 0.4, 'lr': 0.0005},
    {'batch': 128, 'dropout': 0.3, 'lr': 0.003},
]

resultados = []
for i, cfg in enumerate(configuraciones):
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(cfg['dropout']),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])
    m.compile(keras.optimizers.Adam(cfg['lr']), 'sparse_categorical_crossentropy', ['accuracy'])
    h = m.fit(X_tr, y_tr, epochs=10, batch_size=cfg['batch'],
              validation_split=0.1, verbose=0)
    _, acc_te = m.evaluate(X_te, y_te, verbose=0)
    res = {
        'exp': i, 'batch': cfg['batch'], 'dropout': cfg['dropout'], 'lr': cfg['lr'],
        'val_acc': h.history['val_accuracy'][-1], 'test_acc': acc_te
    }
    resultados.append(res)
    print(f"Exp {i}: val_acc={res['val_acc']:.4f}, test_acc={acc_te:.4f}")

df = pd.DataFrame(resultados).sort_values('val_acc', ascending=False)
df.to_csv('experimentos.csv', index=False)
print("\nResultados guardados en experimentos.csv")
print(df.to_string(index=False))

mejor = df.iloc[0]
print(f"\nMejor: batch={int(mejor.batch)}, dropout={mejor.dropout}, lr={mejor.lr}")

# Reentrenar la mejor 30 épocas
tf.random.set_seed(42)
m_final = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(mejor.dropout),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m_final.compile(keras.optimizers.Adam(mejor.lr), 'sparse_categorical_crossentropy', ['accuracy'])
m_final.fit(X_tr, y_tr, epochs=30, batch_size=int(mejor.batch), validation_split=0.1, verbose=0)
_, acc_f = m_final.evaluate(X_te, y_te, verbose=0)
print(f"Mejor configuración, 30 épocas → test_acc={acc_f:.4f}")
```
</details>

---
## Resumen

```python
# TensorBoard
tb = keras.callbacks.TensorBoard(
    log_dir='logs/experimento',
    histogram_freq=1,    # pesos y gradientes
    write_graph=True,
)
# Terminal: tensorboard --logdir logs/

# LR Schedules
lr = keras.optimizers.schedules.CosineDecay(initial_lr=0.001, decay_steps=N)
lr = keras.optimizers.schedules.ExponentialDecay(initial_lr=0.01, decay_steps=1000, decay_rate=0.9)
keras.callbacks.LearningRateScheduler(lambda ep: 0.001 * 0.9**ep)
keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

# Callback personalizado
class MiCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # logs contiene: loss, accuracy, val_loss, val_accuracy
        pass
    def on_train_end(self, logs=None):
        pass

# Diagnóstico
with tf.GradientTape() as tape:
    pred = model(X_batch, training=True)
    loss = loss_fn(y_batch, pred)
grads = tape.gradient(loss, model.trainable_variables)

# Flujo profesional: siempre incluir
EarlyStopping + ModelCheckpoint + TensorBoard + LR schedule
```

## ¿Qué viene en la próxima clase?
En la **Clase 10** (Proyecto final) construiremos un clasificador de imágenes completo con TensorFlow/Keras, aplicando todo lo aprendido en el módulo.